# 02 - Baseline Results Analysis

Analyse der Baseline-Modelle (Majority, Lexicon, Random Forest).

**Inhalte:**
- Baseline-Ergebnisse laden und visualisieren
- Mit BERT-Modellen vergleichen
- Performance-Tabellen erstellen

In [1]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import METRICS_DIR, PLOTS_DIR, LABEL_NAMES
from src.visualize import plot_model_comparison

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Setup complete!')

Setup complete!


## 1. Baseline-Ergebnisse laden

In [2]:
# Baseline Results
baseline_path = METRICS_DIR / 'baseline_results.json'
with open(baseline_path, 'r') as f:
    baseline_data = json.load(f)

# Convert list to DataFrame
baseline_df = pd.DataFrame(baseline_data['baselines'])
baseline_df = baseline_df.sort_values('f1_macro', ascending=False)
baseline_df = baseline_df.set_index('model')

print('Baseline-Modelle:')
print(baseline_df[['f1_macro', 'precision_macro', 'recall_macro', 'accuracy']])

Baseline-Modelle:
                     f1_macro  precision_macro  recall_macro  accuracy
model                                                                 
SVM+TF-IDF           0.623466         0.657724      0.619220  0.696251
Lexikon              0.536628         0.717437      0.566714  0.693907
LogReg+TF-IDF        0.528453         0.677645      0.558988  0.684827
RandomForest+TF-IDF  0.450684         0.765838      0.524695  0.672818
Majority             0.396607         0.328647      0.500000  0.657293


## 2. BERT-Ergebnisse laden

In [3]:
# BERT Results
bert_results = {}

for model_name in ['GBERT', 'mBERT', 'HateBERT']:
    result_path = METRICS_DIR / f'{model_name}_cv_results.json'
    if result_path.exists():
        with open(result_path, 'r') as f:
            data = json.load(f)
            bert_results[model_name] = data

# Als DataFrame
bert_df = pd.DataFrame.from_dict(bert_results, orient='index')
bert_df = bert_df.sort_values('f1_macro_mean', ascending=False)

print('\nBERT-Modelle (5-fold CV):')
print(bert_df[['f1_macro_mean', 'f1_macro_std', 'accuracy_mean', 'accuracy_std']])


BERT-Modelle (5-fold CV):
          f1_macro_mean  f1_macro_std  accuracy_mean  accuracy_std
GBERT          0.792317      0.012029       0.820962      0.008989
mBERT          0.768437      0.015481       0.796490      0.010317
HateBERT       0.672351      0.005797       0.712272      0.010239


## 3. Vergleichstabelle

In [4]:
# Kombinierte Tabelle erstellen
comparison = []

# Baselines
for model, row in baseline_df.iterrows():
    comparison.append({
        'Model': model,
        'F1 (Macro)': f"{row['f1_macro']:.4f}",
        'Precision': f"{row['precision_macro']:.4f}",
        'Recall': f"{row['recall_macro']:.4f}",
        'Accuracy': f"{row['accuracy']:.4f}",
        'Type': 'Baseline'
    })

# BERT
for model, row in bert_df.iterrows():
    comparison.append({
        'Model': model,
        'F1 (Macro)': f"{row['f1_macro_mean']:.4f} ± {row['f1_macro_std']:.4f}",
        'Precision': f"{row['precision_macro_mean']:.4f} ± {row['precision_macro_std']:.4f}",
        'Recall': f"{row['recall_macro_mean']:.4f} ± {row['recall_macro_std']:.4f}",
        'Accuracy': f"{row['accuracy_mean']:.4f} ± {row['accuracy_std']:.4f}",
        'Type': 'BERT'
    })

comp_df = pd.DataFrame(comparison)
print('\n=== Modell-Vergleich ===')
print(comp_df.to_string(index=False))

# Als LaTeX für Paper (optional)
print('\n=== LaTeX Tabelle ===')
print(comp_df.to_latex(index=False))


=== Modell-Vergleich ===
              Model      F1 (Macro)       Precision          Recall        Accuracy     Type
         SVM+TF-IDF          0.6235          0.6577          0.6192          0.6963 Baseline
            Lexikon          0.5366          0.7174          0.5667          0.6939 Baseline
      LogReg+TF-IDF          0.5285          0.6776          0.5590          0.6848 Baseline
RandomForest+TF-IDF          0.4507          0.7658          0.5247          0.6728 Baseline
           Majority          0.3966          0.3286          0.5000          0.6573 Baseline
              GBERT 0.7923 ± 0.0120 0.8043 ± 0.0098 0.7845 ± 0.0141 0.8210 ± 0.0090     BERT
              mBERT 0.7684 ± 0.0155 0.7723 ± 0.0102 0.7661 ± 0.0195 0.7965 ± 0.0103     BERT
           HateBERT 0.6724 ± 0.0058 0.6777 ± 0.0092 0.6716 ± 0.0089 0.7123 ± 0.0102     BERT

=== LaTeX Tabelle ===
\begin{tabular}{llllll}
\toprule
Model & F1 (Macro) & Precision & Recall & Accuracy & Type \\
\midrule
SVM+TF-IDF 

## 4. Visualisierung

In [5]:
# Barplot Vergleich
fig, ax = plt.subplots(figsize=(12, 6))

models = []
f1_values = []
colors = []

# Baselines
for model, row in baseline_df.iterrows():
    models.append(model)
    f1_values.append(row['f1_macro'])
    colors.append('#95a5a6')  # Grau für Baselines

# BERT
for model, row in bert_df.iterrows():
    models.append(model)
    f1_values.append(row['f1_macro_mean'])
    if model == 'GBERT':
        colors.append('#2ecc71')  # Grün für bestes Modell
    else:
        colors.append('#3498db')  # Blau für BERT

bars = ax.bar(range(len(models)), f1_values, color=colors, edgecolor='gray')
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=0)
ax.set_ylabel('F1-Score (Macro)', fontsize=12)
ax.set_title('Modell-Vergleich: Hate Speech Detection', fontsize=14)
ax.grid(True, axis='y', alpha=0.3)

# Werte über Balken
for i, (bar, val) in enumerate(zip(bars, f1_values)):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Users\pauls\AppData\Local\Temp\ipykernel_11200\650462176.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Verbesserung vs. Baseline

In [6]:
# Beste Baseline vs. Beste BERT
best_baseline_f1 = baseline_df['f1_macro'].max()
best_bert_f1 = bert_df['f1_macro_mean'].max()

improvement = best_bert_f1 - best_baseline_f1
improvement_pct = (improvement / best_baseline_f1) * 100

print(f'\n=== Verbesserung durch BERT ===')
print(f'Beste Baseline (Random Forest): F1 = {best_baseline_f1:.4f}')
print(f'Beste BERT (GBERT):             F1 = {best_bert_f1:.4f}')
print(f'Absolute Verbesserung:           {improvement:+.4f}')
print(f'Relative Verbesserung:           {improvement_pct:+.1f}%')


=== Verbesserung durch BERT ===
Beste Baseline (Random Forest): F1 = 0.6235
Beste BERT (GBERT):             F1 = 0.7923
Absolute Verbesserung:           +0.1689
Relative Verbesserung:           +27.1%
